In [ ]:
import warnings
import pandas as pd
import numpy as np
from onekey_algo.custom.components.Radiology import ConventionalRadiomics
from onekey_algo.custom.components.habitat import habitat_feature_fusion
from onekey_algo import get_param_in_cwd
warnings.filterwarnings("ignore")
from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir
import json
from onekey_algo.custom.components.comp1 import icc_filter_features
import os
rad_data = None
mydir = get_param_in_cwd('radio_dir')
icc_thres = 0.7
for modal in ['ICC']:
    rad_data = []
    for icc_round in ['masks', 'ICC']:
        if os.path.exists(f'features/icc{icc_round}_rad_features_{modal}.csv'):
            rad_data_ = pd.read_csv(f'features/icc{icc_round}_rad_features_{modal}.csv', header=0)
        else:
            images, masks = get_image_mask_from_dir(os.path.join(mydir), images=f'images', masks=icc_round)
            param_file = get_param_in_cwd('extractor_settings')
            radiomics = ConventionalRadiomics(param_file, correctMask=True)
            radiomics.extract(images, masks, workers=6, labels=1)
            rad_data_ = radiomics.get_label_data_frame(label=1)
            rad_data_.columns = [f"{c.replace('-', '_')}" if c != 'ID' else 'ID' for c in rad_data_.columns]
            rad_data_.to_csv(f'features/icc{icc_round}_rad_features_{modal}.csv', header=True, index=False)
        rad_data.append(rad_data_)
    rad_data, rad_data_ = rad_data
    joinit_id = pd.merge(rad_data_, rad_data, on='ID', how='inner')[['ID']]
    if not os.path.exists(f'features/icc_{modal}.json'):
        icc = [pd.merge(d, joinit_id, on='ID', how='inner') for d in [rad_data_, rad_data]]        
        icc_sel_features, ficc = icc_filter_features(icc, columns_start=1, with_icc=True, threshold=icc_thres)
        with open(f'features/icc_{modal}.json', 'w') as icc_file:
            print(json.dumps(ficc, ensure_ascii=False, indent=True), file=icc_file)
    else:
        icc = json.loads(open(f'features/icc_{modal}.json').read())
        icc_sel_features = [k for k, v in icc.items() if v > icc_thres]
    print(modal, len(icc_sel_features), 100*len(icc_sel_features)/(rad_data.shape[1] - 1))